# Agenda

### `Features Selection`

- Filter Method

- Wrapper Method

- Embedded Method

# Feaures Selection

Feature selection is a critical step in machine learning that helps in improving model performance by reducing overfitting, improving accuracy, and reducing training time. There are several types of feature selection techniques that we can apply in Python using libraries like scikit-learn. The main types are:

- Filter Method

- Wrapper Method

- Embedded Method

## 1. Filter Method

In the filter method, features are selected based on their statistical significance, often independent of the model.

**Example:** Using Chi-Squared Test for feature selection.

In [2]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
import pandas as pd

# Load dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply SelectKBest using Chi-Squared test
selector = SelectKBest(chi2, k=2)  # Selecting top 2 features
X_new = selector.fit_transform(X, y)

# Get the selected feature columns
selected_columns = X.columns[selector.get_support()]
print(f"Selected features using Filter method: {selected_columns}")


Selected features using Filter method: Index(['petal length (cm)', 'petal width (cm)'], dtype='object')


In [3]:
selected_columns

Index(['petal length (cm)', 'petal width (cm)'], dtype='object')

**For the filter method, the Chi-Squared test gives a score for each feature based on how well it correlates with the target variable**

In [4]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
import pandas as pd

# Load dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply SelectKBest using Chi-Squared test
selector = SelectKBest(chi2, k='all')  # Get scores for all features
selector.fit(X, y)

# Feature scores
scores = pd.DataFrame({'Feature': X.columns, 'Score': selector.scores_})
scores = scores.sort_values(by='Score', ascending=False)

# Top features based on scores
top_features_filter = scores.head(2)  # Let's select top 2 features
print("Filter Method - Feature Scores (Chi-Squared Test):")
print(scores)
print(f"\nTop features selected: {top_features_filter['Feature'].values}")

Filter Method - Feature Scores (Chi-Squared Test):
             Feature       Score
2  petal length (cm)  116.312613
3   petal width (cm)   67.048360
0  sepal length (cm)   10.817821
1   sepal width (cm)    3.710728

Top features selected: ['petal length (cm)' 'petal width (cm)']


### Some Other Filter Methods

- `ANOVA F-test (f_classif)` - For classification problems, assesses the relationship between each feature and the target variable.

- `Mutual Information` (mutual_info_classif / mutual_info_regression) - Measures how much information a feature gives about the target.

- `Variance Threshold` - Removes features with low variance (less informative).



# 1.1. ANOVA F-test (f_classif)

In [5]:
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.svm import SVC
from sklearn.linear_model import LassoCV
import pandas as pd

In [6]:
# 1. Load the Iris dataset for Filter and Wrapper method
iris_data = load_iris()
X_iris = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
y_iris = iris_data.target

# 2. Load California housing dataset for the Embedded method
california_data = fetch_california_housing()
X_california = pd.DataFrame(california_data.data, columns=california_data.feature_names)
y_california = california_data.target

In [7]:
from sklearn.feature_selection import SelectKBest, f_classif

# Apply SelectKBest using ANOVA F-test
anova_selector = SelectKBest(f_classif, k='all')
anova_selector.fit(X_iris, y_iris)

anova_scores = pd.DataFrame({'Feature': X_iris.columns, 'Score': anova_selector.scores_})
anova_scores = anova_scores.sort_values(by='Score', ascending=False)
top_features_anova = anova_scores.head(2)

print("\nFilter Method - Feature Scores (ANOVA F-test):")
print(anova_scores)
print(f"\nTop features selected: {top_features_anova['Feature'].values}")



Filter Method - Feature Scores (ANOVA F-test):
             Feature        Score
2  petal length (cm)  1180.161182
3   petal width (cm)   960.007147
0  sepal length (cm)   119.264502
1   sepal width (cm)    49.160040

Top features selected: ['petal length (cm)' 'petal width (cm)']


# 1.2. Mutual Information (mutual_info_classif)


In [8]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Apply SelectKBest using Mutual Information
mi_selector = SelectKBest(mutual_info_classif, k='all')
mi_selector.fit(X_iris, y_iris)

mi_scores = pd.DataFrame({'Feature': X_iris.columns, 'Score': mi_selector.scores_})
mi_scores = mi_scores.sort_values(by='Score', ascending=False)
top_features_mi = mi_scores.head(2)

print("\nFilter Method - Feature Scores (Mutual Information):")
print(mi_scores)
print(f"\nTop features selected: {top_features_mi['Feature'].values}")



Filter Method - Feature Scores (Mutual Information):
             Feature     Score
3   petal width (cm)  0.985794
2  petal length (cm)  0.978190
0  sepal length (cm)  0.502047
1   sepal width (cm)  0.256073

Top features selected: ['petal width (cm)' 'petal length (cm)']


# 1.3. Variance Threshold

In [9]:
from sklearn.feature_selection import VarianceThreshold

# Apply VarianceThreshold (removes features with low variance)
variance_selector = VarianceThreshold(threshold=0.1)  # Example threshold
variance_selector.fit(X_iris)

variance_selected = pd.DataFrame({'Feature': X_iris.columns, 'Variance': variance_selector.variances_})
variance_selected = variance_selected.sort_values(by='Variance', ascending=False)
top_features_variance = variance_selected.head(2)

print("\nFilter Method - Feature Scores (Variance Threshold):")
print(variance_selected)
print(f"\nTop features selected: {top_features_variance['Feature'].values}")



Filter Method - Feature Scores (Variance Threshold):
             Feature  Variance
2  petal length (cm)  3.095503
0  sepal length (cm)  0.681122
3   petal width (cm)  0.577133
1   sepal width (cm)  0.188713

Top features selected: ['petal length (cm)' 'sepal length (cm)']


# 2. Wrapper Methods

In wrapper methods, subsets of features are selected and evaluated using a predictive model. A popular approach here is Recursive Feature Elimination (RFE).

`Recursive Feature Elimination` with Cross-Validation (RFECV) - Automatically selects the best number of features using cross-validation.

`Sequential Feature Selection` - Iteratively adds/removes features based on cross-validation performance.

`Exhaustive Feature Selection` - Performs a brute-force search over all possible feature subsets.

### Example: Using RFE (Recursive Feature Elimination) with a Support Vector Classifier (SVC)

In [15]:
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.datasets import load_iris
import pandas as pd

# Load dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply RFE with SVC as the estimator
estimator = SVC(kernel="linear")
selector = RFE(estimator, n_features_to_select=2, step=1)
X_new = selector.fit_transform(X, y)

# Get the selected feature columns
selected_columns = X.columns[selector.support_]
print(f"Selected features using Wrapper method: {selected_columns}")


Selected features using Wrapper method: Index(['petal length (cm)', 'petal width (cm)'], dtype='object')


In [16]:
# Print Scores as well

from sklearn.feature_selection import RFE
from sklearn.svm import SVC

# Apply RFE with SVC as the estimator
estimator = SVC(kernel="linear")
selector_rfe = RFE(estimator, n_features_to_select=1)  # Rank all features
selector_rfe.fit(X, y)

# Feature rankings
ranking = pd.DataFrame({'Feature': X.columns, 'Ranking': selector_rfe.ranking_})
ranking = ranking.sort_values(by='Ranking')

# Top features based on rankings
top_features_wrapper = ranking.head(2)  # Let's select top 2 features
print("\nWrapper Method - Feature Rankings (RFE with SVC):")
print(ranking)
print(f"\nTop features selected: {top_features_wrapper['Feature'].values}")

'''
In the wrapper method, Recursive Feature Elimination (RFE) ranks features based on their importance 
as evaluated by the chosen estimator (SVC in this case).
'''


Wrapper Method - Feature Rankings (RFE with SVC):
             Feature  Ranking
2  petal length (cm)        1
3   petal width (cm)        2
1   sepal width (cm)        3
0  sepal length (cm)        4

Top features selected: ['petal length (cm)' 'petal width (cm)']


'\nIn the wrapper method, Recursive Feature Elimination (RFE) ranks features based on their importance \nas evaluated by the chosen estimator (SVC in this case).\n'

# 2.1. Recursive Feature Elimination with Cross-Validation (RFECV)

In [10]:
from sklearn.feature_selection import RFECV

# Apply RFECV with SVC as the estimator
rfecv_selector = RFECV(estimator=SVC(kernel="linear"), step=1, cv=5)
rfecv_selector.fit(X_iris, y_iris)

rfecv_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Ranking': rfecv_selector.ranking_})
rfecv_ranking = rfecv_ranking.sort_values(by='Ranking')
top_features_rfecv = rfecv_ranking.head(2)

print("\nWrapper Method - Feature Rankings (RFECV with SVC):")
print(rfecv_ranking)
print(f"\nTop features selected: {top_features_rfecv['Feature'].values}")



Wrapper Method - Feature Rankings (RFECV with SVC):
             Feature  Ranking
0  sepal length (cm)        1
1   sepal width (cm)        1
2  petal length (cm)        1
3   petal width (cm)        1

Top features selected: ['sepal length (cm)' 'sepal width (cm)']


# 2.2. Sequential Feature Selection

In [11]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

# Apply Sequential Feature Selection with Logistic Regression
sfs_selector = SequentialFeatureSelector(LogisticRegression(), n_features_to_select=2, direction='forward')
sfs_selector.fit(X_iris, y_iris)

sfs_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Ranking': sfs_selector.get_support()})
sfs_ranking = sfs_ranking.sort_values(by='Ranking', ascending=False)
top_features_sfs = sfs_ranking.head(2)

print("\nWrapper Method - Feature Rankings (Sequential Feature Selection):")
print(sfs_ranking)
print(f"\nTop features selected: {top_features_sfs['Feature'].values}")



Wrapper Method - Feature Rankings (Sequential Feature Selection):
             Feature  Ranking
2  petal length (cm)     True
3   petal width (cm)     True
0  sepal length (cm)    False
1   sepal width (cm)    False

Top features selected: ['petal length (cm)' 'petal width (cm)']


# 3. Embedded Methods

Embedded methods perform feature selection during the model training process. These methods are specific to certain algorithms. One common approach is using Lasso Regression (which uses L1 regularization).

`Decision Tree Classifier` - Uses feature importance from a decision tree model.

`Random Forest Classifier` - Uses feature importance from the random forest model.

`Gradient Boosting Classifier` - Uses feature importance from gradient boosting.


## Example: Using Lasso Regression

In [17]:
from sklearn.linear_model import LassoCV
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Load the California housing dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply LassoCV (Lasso with cross-validation)
lasso = LassoCV(cv=5)
lasso.fit(X, y)

# Get the selected feature columns (non-zero coefficients)
selected_columns = X.columns[lasso.coef_ != 0]
print(f"Selected features using Embedded method (Lasso): {selected_columns}")


Selected features using Embedded method (Lasso): Index(['MedInc', 'HouseAge', 'AveRooms', 'Population', 'AveOccup', 'Latitude',
       'Longitude'],
      dtype='object')


In [18]:
from sklearn.linear_model import LassoCV
from sklearn.datasets import fetch_california_housing

# Load the California housing dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Apply LassoCV (Lasso with cross-validation)
lasso = LassoCV(cv=5)
lasso.fit(X, y)

# Feature importance (coefficients)
coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': lasso.coef_})
coefficients = coefficients.sort_values(by='Coefficient', ascending=False)

# Top features based on coefficients
top_features_embedded = coefficients.head(2)  # Let's select top 2 features
print("\nEmbedded Method - Feature Importance (Lasso):")
print(coefficients)
print(f"\nTop features selected: {top_features_embedded['Feature'].values}")
'''
In the embedded method, Lasso regression uses the coefficients as feature importance scores. 
Features with non-zero coefficients are considered important.
'''


Embedded Method - Feature Importance (Lasso):
      Feature  Coefficient
0      MedInc     0.381486
1    HouseAge     0.011287
2    AveRooms     0.002204
4  Population     0.000002
3   AveBedrms     0.000000
5    AveOccup    -0.003513
6    Latitude    -0.339002
7   Longitude    -0.339457

Top features selected: ['MedInc' 'HouseAge']


'\nIn the embedded method, Lasso regression uses the coefficients as feature importance scores. \nFeatures with non-zero coefficients are considered important.\n'

# 3.1. Decision Tree Classifier

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel

# Apply Decision Tree Classifier for feature importance
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_iris, y_iris)

dt_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': dt_clf.feature_importances_})
dt_importance = dt_importance.sort_values(by='Importance', ascending=False)
top_features_dt = dt_importance.head(2)

print("\nEmbedded Method - Feature Importance (Decision Tree):")
print(dt_importance)
print(f"\nTop features selected: {top_features_dt['Feature'].values}")



Embedded Method - Feature Importance (Decision Tree):
             Feature  Importance
2  petal length (cm)    0.564056
3   petal width (cm)    0.422611
0  sepal length (cm)    0.013333
1   sepal width (cm)    0.000000

Top features selected: ['petal length (cm)' 'petal width (cm)']


# 3.2. Random Forest Classifier



In [13]:
from sklearn.ensemble import RandomForestClassifier

# Apply Random Forest Classifier for feature importance
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_iris, y_iris)

rf_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': rf_clf.feature_importances_})
rf_importance = rf_importance.sort_values(by='Importance', ascending=False)
top_features_rf = rf_importance.head(2)

print("\nEmbedded Method - Feature Importance (Random Forest):")
print(rf_importance)
print(f"\nTop features selected: {top_features_rf['Feature'].values}")



Embedded Method - Feature Importance (Random Forest):
             Feature  Importance
2  petal length (cm)    0.436130
3   petal width (cm)    0.436065
0  sepal length (cm)    0.106128
1   sepal width (cm)    0.021678

Top features selected: ['petal length (cm)' 'petal width (cm)']


# 3.3. Gradient Boosting Classifier


In [14]:
from sklearn.ensemble import GradientBoostingClassifier

# Apply Gradient Boosting Classifier for feature importance
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_iris, y_iris)

gb_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': gb_clf.feature_importances_})
gb_importance = gb_importance.sort_values(by='Importance', ascending=False)
top_features_gb = gb_importance.head(2)

print("\nEmbedded Method - Feature Importance (Gradient Boosting):")
print(gb_importance)
print(f"\nTop features selected: {top_features_gb['Feature'].values}")



Embedded Method - Feature Importance (Gradient Boosting):
             Feature  Importance
3   petal width (cm)    0.626406
2  petal length (cm)    0.356272
1   sepal width (cm)    0.011844
0  sepal length (cm)    0.005478

Top features selected: ['petal width (cm)' 'petal length (cm)']


Filter Methods: You can use statistical tests like ANOVA F-test, mutual information, or variance thresholds to assess the relevance of features.

Wrapper Methods: Recursive Feature Elimination with Cross-validation (RFECV), exhaustive feature selection, and sequential feature selection allow you to iteratively select features using a model's performance.

Embedded Methods: Tree-based methods like Decision Tree, Random Forest, and Gradient Boosting automatically provide feature importances during model training, helping select important features.

### Full Example 1

In [19]:
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.svm import SVC
from sklearn.linear_model import LassoCV
import pandas as pd

# 1. Load the Iris dataset for Filter and Wrapper method
iris_data = load_iris()
X_iris = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
y_iris = iris_data.target

# 2. Load California housing dataset for the Embedded method
california_data = fetch_california_housing()
X_california = pd.DataFrame(california_data.data, columns=california_data.feature_names)
y_california = california_data.target

# ------------------- Filter Method (Chi-Squared) -------------------
filter_selector = SelectKBest(chi2, k='all')  # Get scores for all features
filter_selector.fit(X_iris, y_iris)

filter_scores = pd.DataFrame({'Feature': X_iris.columns, 'Score': filter_selector.scores_})
filter_scores = filter_scores.sort_values(by='Score', ascending=False)
top_features_filter = filter_scores.head(2)

# ------------------- Wrapper Method (RFE with SVC) -------------------
wrapper_estimator = SVC(kernel="linear")
wrapper_selector = RFE(wrapper_estimator, n_features_to_select=1)
wrapper_selector.fit(X_iris, y_iris)

wrapper_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Ranking': wrapper_selector.ranking_})
wrapper_ranking = wrapper_ranking.sort_values(by='Ranking')
top_features_wrapper = wrapper_ranking.head(2)

# ------------------- Embedded Method (LassoCV) -------------------
lasso = LassoCV(cv=5)
lasso.fit(X_california, y_california)

lasso_coefficients = pd.DataFrame({'Feature': X_california.columns, 'Coefficient': lasso.coef_})
lasso_coefficients = lasso_coefficients.sort_values(by='Coefficient', ascending=False)
top_features_embedded = lasso_coefficients.head(2)

# ------------------- Display Results -------------------
print("Filter Method - Feature Scores (Chi-Squared Test):")
print(filter_scores)
print(f"\nTop features selected: {top_features_filter['Feature'].values}\n")

print("Wrapper Method - Feature Rankings (RFE with SVC):")
print(wrapper_ranking)
print(f"\nTop features selected: {top_features_wrapper['Feature'].values}\n")

print("Embedded Method - Feature Importance (Lasso):")
print(lasso_coefficients)
print(f"\nTop features selected: {top_features_embedded['Feature'].values}")


Filter Method - Feature Scores (Chi-Squared Test):
             Feature       Score
2  petal length (cm)  116.312613
3   petal width (cm)   67.048360
0  sepal length (cm)   10.817821
1   sepal width (cm)    3.710728

Top features selected: ['petal length (cm)' 'petal width (cm)']

Wrapper Method - Feature Rankings (RFE with SVC):
             Feature  Ranking
2  petal length (cm)        1
3   petal width (cm)        2
1   sepal width (cm)        3
0  sepal length (cm)        4

Top features selected: ['petal length (cm)' 'petal width (cm)']

Embedded Method - Feature Importance (Lasso):
      Feature  Coefficient
0      MedInc     0.381486
1    HouseAge     0.011287
2    AveRooms     0.002204
4  Population     0.000002
3   AveBedrms     0.000000
5    AveOccup    -0.003513
6    Latitude    -0.339002
7   Longitude    -0.339457

Top features selected: ['MedInc' 'HouseAge']


### Full Example 2

In [20]:
import pandas as pd
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, VarianceThreshold, RFECV
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SequentialFeatureSelector
# from mlxtend.feature_selection import ExhaustiveFeatureSelector
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load datasets
iris_data = load_iris()
X_iris = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
y_iris = iris_data.target

california_data = fetch_california_housing()
X_california = pd.DataFrame(california_data.data, columns=california_data.feature_names)
y_california = california_data.target

# ------------------- 1. Filter Methods -------------------

# 1.1. ANOVA F-test
anova_selector = SelectKBest(f_classif, k='all')
anova_selector.fit(X_iris, y_iris)
anova_scores = pd.DataFrame({'Feature': X_iris.columns, 'Score': anova_selector.scores_})
anova_scores = anova_scores.sort_values(by='Score', ascending=False)

# 1.2. Mutual Information
mi_selector = SelectKBest(mutual_info_classif, k='all')
mi_selector.fit(X_iris, y_iris)
mi_scores = pd.DataFrame({'Feature': X_iris.columns, 'Score': mi_selector.scores_})
mi_scores = mi_scores.sort_values(by='Score', ascending=False)

# 1.3. Variance Threshold
variance_selector = VarianceThreshold(threshold=0.1)
variance_selector.fit(X_iris)
variance_selected = pd.DataFrame({'Feature': X_iris.columns, 'Variance': variance_selector.variances_})
variance_selected = variance_selected.sort_values(by='Variance', ascending=False)

# ------------------- 2. Wrapper Methods -------------------

# 2.1. Recursive Feature Elimination with Cross-Validation (RFECV)
rfecv_selector = RFECV(estimator=SVC(kernel="linear"), step=1, cv=5)
rfecv_selector.fit(X_iris, y_iris)
rfecv_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Ranking': rfecv_selector.ranking_})
rfecv_ranking = rfecv_ranking.sort_values(by='Ranking')

# 2.2. Exhaustive Feature Selection
# efs = ExhaustiveFeatureSelector(SVC(kernel="linear"), min_features=1, max_features=3, scoring='accuracy', cv=5)
# efs.fit(X_iris, y_iris)
# efs_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Rank': [i+1 for i in range(len(efs.get_support()))]})
# efs_ranking = efs_ranking.sort_values(by='Rank')

# 2.3. Sequential Feature Selection
sfs_selector = SequentialFeatureSelector(LogisticRegression(), n_features_to_select=2, direction='forward')
sfs_selector.fit(X_iris, y_iris)
sfs_ranking = pd.DataFrame({'Feature': X_iris.columns, 'Ranking': sfs_selector.get_support()})
sfs_ranking = sfs_ranking.sort_values(by='Ranking', ascending=False)

# ------------------- 3. Embedded Methods -------------------

# 3.1. Decision Tree Classifier
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_iris, y_iris)
dt_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': dt_clf.feature_importances_})
dt_importance = dt_importance.sort_values(by='Importance', ascending=False)

# 3.2. Random Forest Classifier
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_iris, y_iris)
rf_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': rf_clf.feature_importances_})
rf_importance = rf_importance.sort_values(by='Importance', ascending=False)

# 3.3. Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_iris, y_iris)
gb_importance = pd.DataFrame({'Feature': X_iris.columns, 'Importance': gb_clf.feature_importances_})
gb_importance = gb_importance.sort_values(by='Importance', ascending=False)

# ------------------- Train and Test AI Models -------------------

# 1. Prepare data (train/test split)
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

# 2. Train Logistic Regression and Random Forest Classifier models

# Logistic Regression model
log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
log_reg_accuracy = accuracy_score(y_test, y_pred_log_reg)

# Random Forest Classifier model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

# ------------------- Display Results -------------------

# 1. Display Filter Methods results
print("\nFilter Method - Feature Scores (ANOVA F-test):")
print(anova_scores)
print("\nFilter Method - Feature Scores (Mutual Information):")
print(mi_scores)
print("\nFilter Method - Feature Scores (Variance Threshold):")
print(variance_selected)

# 2. Display Wrapper Methods results
print("\nWrapper Method - Feature Rankings (RFECV with SVC):")
print(rfecv_ranking)
print("\nWrapper Method - Feature Rankings (Exhaustive Feature Selection):")
# print(efs_ranking)
print("\nWrapper Method - Feature Rankings (Sequential Feature Selection):")
print(sfs_ranking)

# 3. Display Embedded Methods results
print("\nEmbedded Method - Feature Importance (Decision Tree):")
print(dt_importance)
print("\nEmbedded Method - Feature Importance (Random Forest):")
print(rf_importance)
print("\nEmbedded Method - Feature Importance (Gradient Boosting):")
print(gb_importance)

# 4. Display Model Accuracy
print(f"\nLogistic Regression Accuracy: {log_reg_accuracy:.4f}")
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")




Filter Method - Feature Scores (ANOVA F-test):
             Feature        Score
2  petal length (cm)  1180.161182
3   petal width (cm)   960.007147
0  sepal length (cm)   119.264502
1   sepal width (cm)    49.160040

Filter Method - Feature Scores (Mutual Information):
             Feature     Score
2  petal length (cm)  0.989345
3   petal width (cm)  0.988016
0  sepal length (cm)  0.505064
1   sepal width (cm)  0.251912

Filter Method - Feature Scores (Variance Threshold):
             Feature  Variance
2  petal length (cm)  3.095503
0  sepal length (cm)  0.681122
3   petal width (cm)  0.577133
1   sepal width (cm)  0.188713

Wrapper Method - Feature Rankings (RFECV with SVC):
             Feature  Ranking
0  sepal length (cm)        1
1   sepal width (cm)        1
2  petal length (cm)        1
3   petal width (cm)        1

Wrapper Method - Feature Rankings (Exhaustive Feature Selection):

Wrapper Method - Feature Rankings (Sequential Feature Selection):
             Feature  Ranki